# 🏥 Medical Llama 3.1-8B — QLoRA Sequential Training
**Kaggle Notebook | Single LoRA Adapter | Resume-Safe | 3-Dataset Mix**

Datasets (weighted mix):
- 50% `openlifescienceai/medmcqa` — Medical knowledge & facts
- 30% `qiaojin/PubMedQA` — Structured reasoning
- 20% `lavita/ChatDoctor-HealthCareMagic-100k` — Conversational tone

In [ ]:
# ─── 1. Fix Kaggle environment conflicts + Install Dependencies ────────────
# Remove s3fs / gcsfs — they pin fsspec to exact versions that conflict
# with modern ML packages (we don't use either).
!pip uninstall -q -y s3fs gcsfs 2>/dev/null; echo 'Conflict packages removed'

# unsloth requires torch >= 2.11.0 — Kaggle T4 ships 2.10.x.
# We use standard transformers + PEFT + TRL directly (no unsloth API needed).
!pip install -q \
    "transformers>=4.40.0" \
    "peft>=0.10.0" \
    "accelerate>=0.28.0" \
    "bitsandbytes>=0.43.0" \
    "datasets>=2.19.0" \
    "wandb" \
    "huggingface_hub>=0.22.0" \
    "trl>=0.8.6"

print('✅ All packages installed')

In [ ]:
# ─── 2. Imports ────────────────────────────────────────────────────────────
import os
import json
import random
import wandb
import torch
from pathlib import Path
from datasets import load_dataset, concatenate_datasets, Dataset
from transformers import (
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel,
    TaskType,
)
from trl import SFTTrainer
from huggingface_hub import HfApi, login

print(f'PyTorch version : {torch.__version__}')
print(f'GPU available   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name        : {torch.cuda.get_device_name(0)}')

In [ ]:
# ─── 3. Configuration ──────────────────────────────────────────────────────
CFG = {
    # Model
    'base_model'        : 'meta-llama/Llama-3.1-8B-Instruct',
    # ⚠️  Add HF_TOKEN in Kaggle → Add-ons → Secrets, then enable it for this notebook
    'hf_token'          : os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN', ''),
    'hf_repo'           : 'YOUR_HF_USERNAME/medical-llama31-lora',  # your HuggingFace repo

    # Dataset limits per source (adjust for Kaggle RAM)
    'medmcqa_limit'     : 40000,
    'pubmedqa_limit'    : 24000,
    'chatdoctor_limit'  : 16000,

    # Chunk-based sequential training
    'chunk_size'        : 5000,      # samples per Kaggle session chunk
    'chunk_index'       : 0,         # ← INCREMENT this each Kaggle run (0, 1, 2 ...)
    'adapter_path'      : '/kaggle/working/medical_lora',  # local save path
    'resume_from_hub'   : False,     # set True after first run to pull adapter from HF

    # QLoRA
    'lora_r'            : 16,
    'lora_alpha'        : 32,
    'lora_dropout'      : 0.05,
    'target_modules'    : ['q_proj','k_proj','v_proj','o_proj',
                            'gate_proj','up_proj','down_proj'],

    # Training
    'max_seq_length'    : 1024,
    'num_epochs'        : 1,
    'batch_size'        : 4,
    'grad_accum'        : 4,
    'lr'                : 2e-4,
    'warmup_ratio'      : 0.03,
    'weight_decay'      : 0.01,
    'save_steps'        : 200,
    'logging_steps'     : 10,

    # WandB
    'wandb_project'     : 'medical-llama31',
    'wandb_run_name'    : None,   # auto-generated if None
}

# Auto run name
if CFG['wandb_run_name'] is None:
    CFG['wandb_run_name'] = f"chunk_{CFG['chunk_index']:03d}"

print('Configuration loaded:', json.dumps({k: v for k, v in CFG.items() if 'token' not in k}, indent=2))

In [ ]:
# ─── 4. Login to HuggingFace & WandB ──────────────────────────────────────

# ── Validate tokens before attempting login ───────────────────────────────
if not CFG['hf_token']:
    raise ValueError(
        '\n'
        '━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n'
        '  HF_TOKEN is not set!\n'
        '\n'
        '  How to fix:\n'
        '  1. Go to https://huggingface.co/settings/tokens\n'
        '  2. Create a token with READ + WRITE access\n'
        '  3. In Kaggle: Add-ons → Secrets → Add New Secret\n'
        '     Name : HF_TOKEN\n'
        '     Value: hf_xxxxxxxxxxxxxxxxxxxx\n'
        '  4. Click the toggle next to HF_TOKEN to attach it to this notebook\n'
        '  5. Re-run this cell\n'
        '━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━'
    )

wandb_key = os.environ.get('WANDB_API_KEY', '')
if not wandb_key:
    raise ValueError(
        '\n'
        '━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n'
        '  WANDB_API_KEY is not set!\n'
        '\n'
        '  How to fix:\n'
        '  1. Go to https://wandb.ai/settings → API keys\n'
        '  2. Copy your key\n'
        '  3. In Kaggle: Add-ons → Secrets → Add New Secret\n'
        '     Name : WANDB_API_KEY\n'
        '     Value: your_wandb_key\n'
        '  4. Attach it to this notebook, then re-run\n'
        '━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━'
    )

# ── HuggingFace login ──────────────────────────────────────────────────────
login(token=CFG['hf_token'])
print('✅ HuggingFace login successful')

# ── WandB login — single project, all chunks appear under one graph ────────
wandb.login(key=wandb_key)
wandb.init(
    project=CFG['wandb_project'],
    name=CFG['wandb_run_name'],
    config=CFG,
    group='sequential-training',   # groups all chunks in one W&B graph view
    resume='allow',
)
print(f'✅ WandB run started: {wandb.run.name}')

In [ ]:
# ─── 5. Prompt Formatting Functions ────────────────────────────────────────

def format_medmcqa(example):
    """50% Knowledge — MedMCQA: Question + Options → correct answer"""
    options_map = {0: 'A', 1: 'B', 2: 'C', 3: 'D'}
    options_text = (
        f"A) {example.get('opa', '')}\n"
        f"B) {example.get('opb', '')}\n"
        f"C) {example.get('opc', '')}\n"
        f"D) {example.get('opd', '')}"
    )
    correct_idx = example.get('cop', 0)
    correct_letter = options_map.get(correct_idx, 'A')
    correct_text = example.get(f'op{correct_letter.lower()}', '')
    explanation = example.get('exp', '') or ''

    answer = f"{correct_letter}) {correct_text}"
    if explanation.strip():
        answer += f"\n\nExplanation: {explanation.strip()}"

    return {
        'text': (
            f"Question: {example.get('question', '').strip()}\n\n"
            f"Context: {options_text}\n\n"
            f"Answer: {answer}"
        )
    }


def format_pubmedqa(example):
    """30% Reasoning — PubMedQA: abstract context → yes/no/maybe + rationale"""
    contexts = example.get('context', {}).get('contexts', [])
    context_text = ' '.join(contexts)[:800] if contexts else 'No context provided.'
    answer = example.get('final_decision', '').strip()
    long_answer = example.get('long_answer', '').strip()
    full_answer = f"{answer.capitalize()}."
    if long_answer:
        full_answer += f" {long_answer}"

    return {
        'text': (
            f"Question: {example.get('question', '').strip()}\n\n"
            f"Context: {context_text}\n\n"
            f"Answer: {full_answer}"
        )
    }


def format_chatdoctor(example):
    """20% Chat Style — ChatDoctor: user question → doctor response"""
    patient = example.get('input', '').strip()
    doctor = example.get('output', '').strip()
    if not patient or not doctor:
        return {'text': ''}
    return {
        'text': (
            f"User: {patient}\n\n"
            f"Doctor: {doctor}"
        )
    }


print('✅ Prompt formatting functions defined')

In [ ]:
# ─── 6. Load & Format Datasets ─────────────────────────────────────────────
print('Loading datasets ...')

# --- MedMCQA (50%) ---
ds_medmcqa_raw = load_dataset('openlifescienceai/medmcqa', split='train')
ds_medmcqa_raw = ds_medmcqa_raw.shuffle(seed=42).select(range(CFG['medmcqa_limit']))
ds_medmcqa = ds_medmcqa_raw.map(format_medmcqa, remove_columns=ds_medmcqa_raw.column_names)
ds_medmcqa = ds_medmcqa.filter(lambda x: len(x['text']) > 50)
print(f'  MedMCQA  : {len(ds_medmcqa):,} samples')

# --- PubMedQA (30%) ---
ds_pubmed_raw = load_dataset('qiaojin/PubMedQA', 'pqa_labeled', split='train')
ds_pubmed_raw = ds_pubmed_raw.shuffle(seed=42).select(range(min(CFG['pubmedqa_limit'], len(ds_pubmed_raw))))
ds_pubmed = ds_pubmed_raw.map(format_pubmedqa, remove_columns=ds_pubmed_raw.column_names)
ds_pubmed = ds_pubmed.filter(lambda x: len(x['text']) > 50)
print(f'  PubMedQA : {len(ds_pubmed):,} samples')

# --- ChatDoctor (20%) ---
ds_chat_raw = load_dataset('lavita/ChatDoctor-HealthCareMagic-100k', split='train')
ds_chat_raw = ds_chat_raw.shuffle(seed=42).select(range(CFG['chatdoctor_limit']))
ds_chat = ds_chat_raw.map(format_chatdoctor, remove_columns=ds_chat_raw.column_names)
ds_chat = ds_chat.filter(lambda x: len(x['text']) > 50)
print(f'  ChatDoctor: {len(ds_chat):,} samples')

# --- Combine with weighted interleaving ---
all_samples = (
    ds_medmcqa.to_list() +
    ds_pubmed.to_list() +
    ds_chat.to_list()
)
random.seed(42)
random.shuffle(all_samples)
combined_dataset = Dataset.from_list(all_samples)

print(f'\nTotal combined dataset: {len(combined_dataset):,} samples')

In [ ]:
# ─── 7. Extract Current Chunk ──────────────────────────────────────────────
chunk_start = CFG['chunk_index'] * CFG['chunk_size']
chunk_end   = min(chunk_start + CFG['chunk_size'], len(combined_dataset))

if chunk_start >= len(combined_dataset):
    raise ValueError(f'chunk_index {CFG["chunk_index"]} is out of range! '
                     f'Max index = {len(combined_dataset) // CFG["chunk_size"]}')

train_chunk = combined_dataset.select(range(chunk_start, chunk_end))
print(f'Training chunk {CFG["chunk_index"]}: samples {chunk_start:,} → {chunk_end:,} '
      f'({len(train_chunk):,} total)')

total_chunks = (len(combined_dataset) + CFG['chunk_size'] - 1) // CFG['chunk_size']
print(f'Progress: chunk {CFG["chunk_index"]} of {total_chunks - 1} '
      f'({100*(CFG["chunk_index"]+1)/total_chunks:.1f}% complete)')

In [ ]:
# ─── 8. Load Tokenizer ─────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(
    CFG['base_model'],
    token=CFG['hf_token'],
    trust_remote_code=True,
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
print('✅ Tokenizer loaded')

In [ ]:
# ─── 9. Load Base Model with QLoRA (4-bit) ─────────────────────────────────
from transformers import AutoModelForCausalLM

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    CFG['base_model'],
    token=CFG['hf_token'],
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
base_model.config.use_cache = False
base_model.config.pretraining_tp = 1
print('✅ Base model loaded with 4-bit quantization')

In [ ]:
# ─── 10. Attach / Resume LoRA Adapter ──────────────────────────────────────
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=CFG['lora_r'],
    lora_alpha=CFG['lora_alpha'],
    lora_dropout=CFG['lora_dropout'],
    target_modules=CFG['target_modules'],
    bias='none',
)

if CFG['resume_from_hub']:
    # Pull existing adapter from HuggingFace to continue training
    print(f'Resuming adapter from HuggingFace: {CFG["hf_repo"]} ...')
    model = PeftModel.from_pretrained(
        base_model,
        CFG['hf_repo'],
        token=CFG['hf_token'],
        is_trainable=True,
    )
    print('✅ Adapter resumed from HuggingFace')
elif Path(CFG['adapter_path']).exists():
    # Resume from local checkpoint (same Kaggle session)
    print(f'Resuming adapter from local: {CFG["adapter_path"]} ...')
    model = PeftModel.from_pretrained(
        base_model,
        CFG['adapter_path'],
        is_trainable=True,
    )
    print('✅ Adapter resumed from local checkpoint')
else:
    # First run — create fresh LoRA adapter
    print('Creating new LoRA adapter (first run) ...')
    model = get_peft_model(base_model, lora_config)
    print('✅ New LoRA adapter created')

model.print_trainable_parameters()

In [ ]:
# ─── 11. Training Arguments ────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=CFG['adapter_path'],
    num_train_epochs=CFG['num_epochs'],
    per_device_train_batch_size=CFG['batch_size'],
    gradient_accumulation_steps=CFG['grad_accum'],
    learning_rate=CFG['lr'],
    warmup_ratio=CFG['warmup_ratio'],
    weight_decay=CFG['weight_decay'],
    fp16=False,
    bf16=True,
    logging_steps=CFG['logging_steps'],
    save_steps=CFG['save_steps'],
    save_total_limit=2,
    report_to='wandb',
    run_name=CFG['wandb_run_name'],
    lr_scheduler_type='cosine',
    optim='paged_adamw_8bit',
    dataloader_num_workers=0,
    group_by_length=True,
    seed=42,
)
print('✅ Training arguments configured')

In [ ]:
# ─── 12. SFT Trainer ───────────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_chunk,
    dataset_text_field='text',
    max_seq_length=CFG['max_seq_length'],
    args=training_args,
    packing=False,
)
print(f'✅ Trainer ready | chunk {CFG["chunk_index"]} | {len(train_chunk):,} samples')

In [ ]:
# ─── 13. Train ─────────────────────────────────────────────────────────────
print(f'🚀 Starting training on chunk {CFG["chunk_index"]} ...')
trainer.train()
print('✅ Training complete')

In [ ]:
# ─── 14. Save Adapter Locally ──────────────────────────────────────────────
model.save_pretrained(CFG['adapter_path'])
tokenizer.save_pretrained(CFG['adapter_path'])
print(f'✅ Adapter saved locally: {CFG["adapter_path"]}')

# Verify required files exist
required = ['adapter_config.json', 'adapter_model.safetensors']
for f in required:
    exists = Path(CFG['adapter_path'], f).exists()
    print(f'  {f}: {"✅" if exists else "❌ MISSING"}')

In [ ]:
# ─── 15. Push Adapter to HuggingFace ──────────────────────────────────────
api = HfApi()

# Create repo if it doesn't exist
try:
    api.create_repo(repo_id=CFG['hf_repo'], token=CFG['hf_token'], exist_ok=True, private=True)
    print(f'✅ HuggingFace repo ready: {CFG["hf_repo"]}')
except Exception as e:
    print(f'Repo creation note: {e}')

# Upload adapter files
model.push_to_hub(
    CFG['hf_repo'],
    token=CFG['hf_token'],
    commit_message=f'Sequential training — chunk {CFG["chunk_index"]}',
)
tokenizer.push_to_hub(
    CFG['hf_repo'],
    token=CFG['hf_token'],
)
print(f'✅ Adapter uploaded to HuggingFace: https://huggingface.co/{CFG["hf_repo"]}')

# Finish WandB run
wandb.finish()
print('✅ WandB run finished')
print()
print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'  Chunk {CFG["chunk_index"]} DONE')
print(f'  Next: set chunk_index = {CFG["chunk_index"]+1}')
print(f'         set resume_from_hub = True')
print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')